# TypeNet — Full Aalto 136M Keystroke Dataset Training

**Two-phase notebook:**
1. **Preprocess** `Keystrokes.zip` → `aalto_full.h5` (run once, ~30–60 min)
2. **Train** TypeNet with triplet loss from the HDF5 file (saves best model to Drive)

**Before running:**
- Upload `Keystrokes.zip` (1.5 GB) to your Google Drive, e.g. `My Drive/typenet/Keystrokes.zip`
- Set **Runtime → Change runtime type → GPU** (T4 is fine, A100 is faster)


## 1. Set Up Google Colab Environment

In [ ]:
import torch
import sys

# Check GPU
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU     : {gpu.name}  ({gpu.total_memory // 1024**3} GB VRAM)")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/My Drive/typenet'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive folder: {DRIVE_BASE}")
print("Files already there:", os.listdir(DRIVE_BASE) if os.path.exists(DRIVE_BASE) else [])


## 3. Install Required Libraries

In [ ]:
!pip install -q h5py tqdm
import h5py
import tqdm
print("h5py   :", h5py.__version__)
print("tqdm   :", tqdm.__version__)


## 4. Configuration — Paths & Hyperparameters

Edit the paths here if your Drive layout is different.

In [ ]:
# ── Paths (edit if needed) ─────────────────────────────────────────────────────
DRIVE_BASE      = '/content/drive/My Drive/typenet'

ZIP_PATH        = f'{DRIVE_BASE}/Keystrokes.zip'   # upload here before running
H5_PATH         = f'{DRIVE_BASE}/aalto_full.h5'    # created by preprocessing step
MODEL_PATH      = f'{DRIVE_BASE}/typenet_pretrained.pth'
CHECKPOINT_PATH = f'{DRIVE_BASE}/typenet.ckpt.pth'

# ── Preprocessing config ────────────────────────────────────────────────────────
# NOTE: The Aalto dataset sentences are short (~40–55 keystrokes each).
# SEQ_LEN=70 was too long — most sentences produced zero windows, dropping ~90%
# of users.  SEQ_LEN=30 / STRIDE=15 fits comfortably within each sentence and
# gives ~15–25 sequences per user across the 15 typed sentences.
SEQ_LEN   = 30    # keystroke window length (reduced from 70 for short Aalto sentences)
STRIDE    = 15    # sliding-window stride (50% overlap)
MIN_SEQS  = 5     # discard users with fewer valid sequences
MAX_SEQS  = 50    # cap sequences per user for balanced triplet sampling

# Clipping bounds for raw millisecond values → normalise each feature to [0,1]
CLIP = {
    'HL': (0,    1000),   # hold latency
    'IL': (-200, 2000),   # inter-key latency (DD)
    'PL': (-500, 2000),   # press latency    (UD)
    'RL': (-500, 2000),   # release latency  (UU)
}

# ── Dataset split (per-user — never leak users across splits) ──────────────────
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Ratios must sum to 1"
SPLIT_SEED  = 42

# ── TypeNet hyperparameters (paper values) ──────────────────────────────────────
INPUT_SIZE   = 5      # [HL, IL, PL, RL, KeyCode]
HIDDEN_SIZE  = 128
OUTPUT_SIZE  = 128
DROPOUT_RATE = 0.5

EPOCHS             = 100
BATCH_SIZE         = 512
LEARNING_RATE      = 0.005
MARGIN             = 1.5
TRIPLETS_PER_USER  = 10
NUM_WORKERS        = 2
CHECKPOINT_EVERY   = 5

import os
import zipfile
assert os.path.exists(ZIP_PATH), \
    f"Keystrokes.zip not found at {ZIP_PATH}.\nUpload it to Drive first."
print("✅ Keystrokes.zip found:", ZIP_PATH)
print(f"   Size: {os.path.getsize(ZIP_PATH)/1e9:.2f} GB")
print(f"\nDataset split  →  train {TRAIN_RATIO*100:.0f}%  |  "
      f"val {VAL_RATIO*100:.0f}%  |  test {TEST_RATIO*100:.0f}%  (per-user)")
print(f"SEQ_LEN={SEQ_LEN}  STRIDE={STRIDE}  MIN_SEQS={MIN_SEQS}  MAX_SEQS={MAX_SEQS}")


## 5. Preprocess the Full Dataset  
**Run this section only once.** It reads every user file inside `Keystrokes.zip`, extracts TypeNet features using a sliding window, and writes the result to `aalto_full.h5` on Drive (~2–4 GB, ~30–90 min).  
If `aalto_full.h5` already exists on Drive the cell will skip preprocessing automatically.

In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import h5py
from tqdm.auto import tqdm

# ──────────────────────────────────────────────────────────────────────────────
# Feature helpers
# ──────────────────────────────────────────────────────────────────────────────
def _norm(value, lo, hi):
    """Clip then scale to [0, 1]."""
    return max(0.0, min(1.0, (float(value) - lo) / (hi - lo)))


def extract_sequences(lines, seq_len, stride):
    """
    Parse one user's raw byte lines from the Aalto dataset.
    Group keystrokes by TEST_SECTION_ID (sentence), compute TypeNet features
    [HL, IL, PL, RL, KC], then emit sliding windows of length seq_len.

    Returns list of float32 arrays shaped (seq_len, 5).
    """
    sentences = {}   # section_id → [(press_ms, release_ms, keycode)]

    for raw in lines:
        try:
            row = raw.decode('utf-8', errors='replace').rstrip('\n').split('\t')
        except Exception:
            continue
        if len(row) < 9:
            continue
        if row[0].strip() == 'PARTICIPANT_ID':
            continue   # header row

        try:
            section_id   = row[1].strip()
            press_time   = float(row[5].strip())
            release_time = float(row[6].strip())
            keycode      = int(row[8].strip())
        except (ValueError, IndexError):
            continue

        if press_time <= 0 or release_time < press_time:
            continue

        sentences.setdefault(section_id, []).append(
            (press_time, release_time, keycode)
        )

    all_seqs = []
    for keystrokes in sentences.values():
        keystrokes.sort(key=lambda x: x[0])
        n = len(keystrokes)
        if n < seq_len:
            continue

        feat = np.zeros((n, 5), dtype=np.float32)
        for i, (press_i, release_i, kc_i) in enumerate(keystrokes):
            feat[i, 0] = _norm(release_i - press_i,              *CLIP['HL'])
            feat[i, 4] = float(kc_i) / 256.0
            if i > 0:
                press_prev, release_prev, _ = keystrokes[i - 1]
                feat[i, 1] = _norm(press_i   - press_prev,   *CLIP['IL'])
                feat[i, 2] = _norm(press_i   - release_prev, *CLIP['PL'])
                feat[i, 3] = _norm(release_i - release_prev, *CLIP['RL'])

        start = 0
        while start + seq_len <= n:
            all_seqs.append(feat[start : start + seq_len].copy())
            start += stride

    return all_seqs


# ──────────────────────────────────────────────────────────────────────────────
# Main preprocessing pipeline
# ──────────────────────────────────────────────────────────────────────────────
CHUNK_WRITE = 4096   # flush buffer every N sequences

if os.path.exists(H5_PATH):
    with h5py.File(H5_PATH, 'r') as hf:
        n_u = int(hf.attrs.get('n_users',     0))
        n_s = int(hf.attrs.get('n_sequences', 0))
    print("⏭️  aalto_full.h5 already exists — skipping preprocessing.")
    print(f"   Users: {n_u:,}   Sequences: {n_s:,}")
else:
    np.random.seed(42)

    # Discover user files
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        user_entries = sorted(
            e for e in zf.namelist()
            if e.endswith('_keystrokes.txt') and 'metadata' not in e
        )
    total_files = len(user_entries)
    print(f"Found {total_files:,} user files in ZIP.\nStarting preprocessing …\n")

    t_start = time.time()

    with zipfile.ZipFile(ZIP_PATH, 'r') as zf, \
         h5py.File(H5_PATH, 'w') as hf:

        seq_ds = hf.create_dataset(
            'sequences',
            shape=(0, SEQ_LEN, 5), maxshape=(None, SEQ_LEN, 5),
            dtype='float32',
            chunks=(min(CHUNK_WRITE, 2048), SEQ_LEN, 5),
            compression='lzf',
        )
        lbl_ds = hf.create_dataset(
            'labels',
            shape=(0,), maxshape=(None,),
            dtype='int32',
            chunks=(min(CHUNK_WRITE * 4, 8192),),
            compression='lzf',
        )

        seq_buf, lbl_buf       = [], []
        orig_ids, index_map    = [], []
        total_seqs, user_idx   = 0, 0

        def flush():
            global seq_buf, lbl_buf
            if not seq_buf:
                return
            batch = np.stack(seq_buf, axis=0)
            old_n, new_n = seq_ds.shape[0], seq_ds.shape[0] + len(batch)
            seq_ds.resize(new_n, axis=0)
            lbl_ds.resize(new_n, axis=0)
            seq_ds[old_n:new_n] = batch
            lbl_ds[old_n:new_n] = np.array(lbl_buf, dtype=np.int32)
            seq_buf.clear()
            lbl_buf.clear()

        for entry in tqdm(user_entries, desc='Processing users', unit='user'):
            try:
                raw_bytes = zf.read(entry)
            except Exception as e:
                tqdm.write(f'WARN: {entry}: {e}')
                continue

            seqs = extract_sequences(raw_bytes.splitlines(), SEQ_LEN, STRIDE)
            if len(seqs) < MIN_SEQS:
                continue

            if len(seqs) > MAX_SEQS:
                idxs = np.random.choice(len(seqs), MAX_SEQS, replace=False)
                seqs = [seqs[i] for i in sorted(idxs)]

            try:
                orig_id = int(Path(entry).name.split('_')[0])
            except ValueError:
                orig_id = -1

            n_seqs = len(seqs)
            orig_ids.append(orig_id)
            index_map.append((total_seqs, total_seqs + n_seqs))
            total_seqs += n_seqs

            for s in seqs:
                seq_buf.append(s)
                lbl_buf.append(user_idx)
            user_idx += 1

            if len(seq_buf) >= CHUNK_WRITE:
                flush()

        flush()

        hf.create_dataset('orig_ids',  data=np.array(orig_ids,  dtype=np.int32))
        hf.create_dataset('index_map', data=np.array(index_map, dtype=np.int64))
        hf.attrs['n_users']     = user_idx
        hf.attrs['n_sequences'] = total_seqs
        hf.attrs['seq_len']     = SEQ_LEN
        hf.attrs['stride']      = STRIDE

    elapsed = time.time() - t_start
    size_gb = os.path.getsize(H5_PATH) / 1e9
    print(f"\n✅ Preprocessing complete in {elapsed/60:.1f} min")
    print(f"   Valid users  : {user_idx:,}")
    print(f"   Total seqs   : {total_seqs:,}")
    print(f"   HDF5 size    : {size_gb:.2f} GB")
    print(f"   Saved to     : {H5_PATH}")


## 5b. Copy HDF5 to Local Colab SSD  
**Critical for speed.** Reading from Google Drive is ~20× slower than Colab's local SSD.  
This cell copies `aalto_full.h5` to `/content/` (takes ~3–5 min once) and redirects `H5_PATH` so all training reads happen locally.  
The Drive copy is kept as backup — only the read path changes.

In [ ]:
import shutil
import time as _t

LOCAL_H5 = '/content/aalto_full.h5'

if not os.path.exists(LOCAL_H5):
    print(f"Copying {H5_PATH}  →  {LOCAL_H5}")
    print("(~1.25 GB — takes ~3–5 min on a typical Colab session)\n")
    _t0 = _t.time()
    shutil.copy2(H5_PATH, LOCAL_H5)
    elapsed_copy = _t.time() - _t0
    size_gb = os.path.getsize(LOCAL_H5) / 1e9
    print(f"✅ Copy complete in {elapsed_copy/60:.1f} min  ({size_gb:.2f} GB)")
else:
    print(f"✅ Local HDF5 already exists: {LOCAL_H5}  "
          f"({os.path.getsize(LOCAL_H5)/1e9:.2f} GB)")

# Redirect H5_PATH — all downstream cells (dataset, training, eval) use this
H5_PATH = LOCAL_H5
print(f"\nH5_PATH → {H5_PATH}  (local SSD — fast I/O)")


## 6. Define the Model, Dataset, and Loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ──────────────────────────────────────────────────────────────────────────────
# RAM-backed Dataset — loads all sequences into RAM once, then indexes instantly
# ──────────────────────────────────────────────────────────────────────────────
class HDF5TripletDataset(Dataset):
    """
    Loads the full sequences array from HDF5 into a numpy array at init time,
    OR accepts a pre-loaded array to avoid loading twice for train/val splits.
    __getitem__ does pure numpy indexing — no file I/O on the hot path.

    Memory: ~2.7 GB for the full 4.5M-sequence dataset — fits on T4 (12 GB RAM).

    Parameters
    ----------
    h5_path           : path to aalto_full.h5
    user_indices      : 1-D int array of user indices to include (None = all).
    triplets_per_user : triplets sampled per user per epoch.
    sequences         : pre-loaded numpy array (N, SEQ_LEN, 5) — pass this to
                        avoid reloading the 2.7 GB array for each split.
    """

    def __init__(self, h5_path, user_indices=None, triplets_per_user=10,
                 sequences=None):
        self.triplets_per_user = triplets_per_user

        with h5py.File(h5_path, 'r') as hf:
            total_users    = int(hf.attrs['n_users'])
            full_index_map = hf['index_map'][:]          # (total_users, 2)
            if sequences is None:
                print(f"  Loading sequences into RAM from {h5_path} …", flush=True)
                t0 = __import__('time').time()
                self.sequences = hf['sequences'][:]      # (N, SEQ_LEN, 5) float32
                elapsed = __import__('time').time() - t0
                size_gb = self.sequences.nbytes / 1e9
                print(f"  ✅ Loaded {self.sequences.shape[0]:,} sequences "
                      f"({size_gb:.2f} GB) in {elapsed:.1f}s", flush=True)
            else:
                self.sequences = sequences               # shared — no copy

        if user_indices is None:
            user_indices = np.arange(total_users, dtype=np.int64)

        self.user_indices = np.asarray(user_indices, dtype=np.int64)
        self.index_map    = full_index_map[self.user_indices]  # (n_users, 2)
        self.n_users      = len(self.user_indices)

        print(f"  [Dataset] subset_users={self.n_users:,}  "
              f"triplets/epoch≈{self.n_users * triplets_per_user:,}", flush=True)

    def __len__(self):
        return self.n_users * self.triplets_per_user

    def __getitem__(self, idx):
        anc_user        = idx % self.n_users
        a_start, a_end  = self.index_map[anc_user]
        a_count         = int(a_end - a_start)

        picks = np.random.choice(a_count, 2, replace=(a_count < 2))
        a_idx = int(a_start) + picks[0]
        p_idx = int(a_start) + picks[1]

        neg_user = np.random.randint(0, self.n_users)
        while neg_user == anc_user:
            neg_user = np.random.randint(0, self.n_users)
        n_start, n_end = self.index_map[neg_user]
        n_idx = int(n_start) + np.random.randint(0, int(n_end - n_start))

        return (
            torch.from_numpy(self.sequences[a_idx]),
            torch.from_numpy(self.sequences[p_idx]),
            torch.from_numpy(self.sequences[n_idx]),
        )


# ──────────────────────────────────────────────────────────────────────────────
# TypeNet Architecture
# ──────────────────────────────────────────────────────────────────────────────
class TypeNet(nn.Module):
    """
    Two stacked LSTMs with BatchNorm, Dropout, and a linear projection.
    Outputs L2-normalised 128-d embeddings.
    """

    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(INPUT_SIZE,  HIDDEN_SIZE, batch_first=True)
        self.bn1   = nn.BatchNorm1d(HIDDEN_SIZE)
        self.drop1 = nn.Dropout(DROPOUT_RATE)

        self.lstm2 = nn.LSTM(HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)
        self.bn2   = nn.BatchNorm1d(HIDDEN_SIZE)
        self.drop2 = nn.Dropout(DROPOUT_RATE)

        self.fc    = nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)

    def forward_one(self, x):
        out, _ = self.lstm1(x)
        out    = self.bn1(out.permute(0, 2, 1)).permute(0, 2, 1)
        out    = self.drop1(out)
        out, _ = self.lstm2(out)
        out    = self.bn2(out.permute(0, 2, 1)).permute(0, 2, 1)
        out    = self.drop2(out)
        return F.normalize(self.fc(out[:, -1, :]), p=2, dim=1)

    def forward(self, a, p, n):
        return self.forward_one(a), self.forward_one(p), self.forward_one(n)


# ──────────────────────────────────────────────────────────────────────────────
# Triplet Loss
# ──────────────────────────────────────────────────────────────────────────────
class TripletLoss(nn.Module):
    def __init__(self, margin=MARGIN):
        super().__init__()
        self.margin = margin

    def forward(self, a, p, n):
        d_pos = (a - p).pow(2).sum(1)
        d_neg = (a - n).pow(2).sum(1)
        return F.relu(d_pos - d_neg + self.margin).mean()


# ──────────────────────────────────────────────────────────────────────────────
# User-level split  (80 / 10 / 10)
# ──────────────────────────────────────────────────────────────────────────────
def make_user_splits(h5_path, train_ratio, val_ratio, seed):
    """
    Shuffle all user indices with a fixed seed and split them into
    train / val / test index arrays.  The split is deterministic so
    checkpoints and re-runs always reference the same users.
    """
    with h5py.File(h5_path, 'r') as hf:
        n_total = int(hf.attrs['n_users'])

    rng = np.random.default_rng(seed)
    idx = rng.permutation(n_total)

    n_train = int(n_total * train_ratio)
    n_val   = int(n_total * val_ratio)

    train_idx = idx[:n_train]
    val_idx   = idx[n_train : n_train + n_val]
    test_idx  = idx[n_train + n_val :]

    print(f"Total users : {n_total:,}")
    print(f"  train     : {len(train_idx):,}  ({len(train_idx)/n_total*100:.1f}%)")
    print(f"  val       : {len(val_idx):,}   ({len(val_idx)/n_total*100:.1f}%)")
    print(f"  test      : {len(test_idx):,}   ({len(test_idx)/n_total*100:.1f}%)")
    return train_idx, val_idx, test_idx


print("Model, dataset, and loss classes defined ✅")


## 7. Train the Model  
The training loop uses **cosine-annealing LR**, **gradient clipping**, automatic **best-model saving**, and **resumable checkpoints** — so if the Colab session disconnects you can pick up where you left off.

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}\n")

# ── User-level 80/10/10 split ──────────────────────────────────────────────────
train_idx, val_idx, test_idx = make_user_splits(
    H5_PATH, TRAIN_RATIO, VAL_RATIO, SPLIT_SEED
)

# Save the split so evaluation / inference can reuse the exact same test users
np.save(DRIVE_BASE + '/split_train.npy', train_idx)
np.save(DRIVE_BASE + '/split_val.npy',   val_idx)
np.save(DRIVE_BASE + '/split_test.npy',  test_idx)
print("Splits saved to Drive.\n")

# ── Load sequences into RAM once — shared by train and val ────────────────────
print("Loading sequences into RAM (shared by train + val) …")
import time as _time
_t0 = _time.time()
with h5py.File(H5_PATH, 'r') as _hf:
    _shared_seqs = _hf['sequences'][:]
print(f"  ✅ {_shared_seqs.shape[0]:,} sequences  "
      f"({_shared_seqs.nbytes/1e9:.2f} GB)  loaded in {_time.time()-_t0:.1f}s\n")

# ── Build train / val datasets (pass shared array — no second load) ───────────
print("Train dataset:")
train_ds = HDF5TripletDataset(H5_PATH, user_indices=train_idx,
                               triplets_per_user=TRIPLETS_PER_USER,
                               sequences=_shared_seqs)
print("Val dataset:")
val_ds   = HDF5TripletDataset(H5_PATH, user_indices=val_idx,
                               triplets_per_user=TRIPLETS_PER_USER,
                               sequences=_shared_seqs)

train_loader = DataLoader(
    train_ds,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    num_workers     = 0,      # data is in RAM — workers would copy 2.7 GB each
    pin_memory      = (device.type == 'cuda'),
)
val_loader = DataLoader(
    val_ds,
    batch_size = BATCH_SIZE,
    shuffle    = False,
    num_workers     = 0,
    pin_memory      = (device.type == 'cuda'),
)

# ── Model & optimiser ──────────────────────────────────────────────────────────
model     = TypeNet().to(device)
criterion = TripletLoss(margin=MARGIN)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.01
)

# ── Resume from checkpoint if available ────────────────────────────────────────
start_epoch   = 0
best_val_loss = float('inf')
train_history = []    # avg train loss per epoch
val_history   = []    # avg val   loss per epoch

if os.path.exists(CHECKPOINT_PATH):
    print(f"\nLoading checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch   = ckpt['epoch']
    best_val_loss = ckpt.get('best_val_loss', best_val_loss)
    train_history = ckpt.get('train_history', [])
    val_history   = ckpt.get('val_history',   [])
    print(f"Resumed from epoch {start_epoch}  best_val_loss={best_val_loss:.4f}")
else:
    print("No checkpoint found — starting fresh.")

# ── Training loop ──────────────────────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>10}  {'LR':>10}  {'Time':>6}")
print(f"{'─'*65}")

for epoch in range(start_epoch, EPOCHS):
    # ── Train ──────────────────────────────────────────────────
    model.train()
    total_train = 0.0
    t0          = time.time()

    bar = tqdm(train_loader, desc=f'Train {epoch+1:>3}/{EPOCHS}', leave=False)
    for anchor, pos, neg in bar:
        anchor, pos, neg = anchor.to(device), pos.to(device), neg.to(device)
        optimizer.zero_grad()
        e_a, e_p, e_n = model(anchor, pos, neg)
        loss = criterion(e_a, e_p, e_n)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_train += loss.item()
        bar.set_postfix(loss=f'{loss.item():.4f}')

    # ── Validation ─────────────────────────────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for anchor, pos, neg in val_loader:
            anchor, pos, neg = anchor.to(device), pos.to(device), neg.to(device)
            e_a, e_p, e_n = model(anchor, pos, neg)
            total_val += criterion(e_a, e_p, e_n).item()

    scheduler.step()

    avg_train = total_train / len(train_loader)
    avg_val   = total_val   / len(val_loader)
    elapsed   = time.time() - t0
    lr_now    = scheduler.get_last_lr()[0]

    train_history.append(avg_train)
    val_history.append(avg_val)

    marker = ' ← best' if avg_val < best_val_loss else ''
    print(f"{epoch+1:>6}  {avg_train:>11.4f}  {avg_val:>10.4f}  "
          f"{lr_now:>10.6f}  {elapsed:>4.0f}s{marker}")

    # Save best model (judged by validation loss)
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), MODEL_PATH)

    # Periodic checkpoint for resuming
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        torch.save({
            'epoch':        epoch + 1,
            'model':        model.state_dict(),
            'optimizer':    optimizer.state_dict(),
            'scheduler':    scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'train_history': train_history,
            'val_history':   val_history,
        }, CHECKPOINT_PATH)
        print(f"  ✓ Checkpoint → {CHECKPOINT_PATH}")

print(f"\nTraining complete.  Best val loss: {best_val_loss:.4f}")
print(f"Best model saved to: {MODEL_PATH}")


## 8. Evaluate — Loss Curve & Embedding Sanity Check

In [ ]:
import matplotlib.pyplot as plt

# ── 1. Train vs Val loss curves ────────────────────────────────────────────────
if train_history:
    fig, ax = plt.subplots(figsize=(11, 4))
    epochs  = range(1, len(train_history) + 1)
    ax.plot(epochs, train_history, label='Train (80%)',      marker='o', ms=3)
    ax.plot(epochs, val_history,   label='Validation (10%)', marker='s', ms=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Avg Triplet Loss')
    ax.set_title('TypeNet Training & Validation Loss — Full Aalto Dataset\n'
                 '(Split: 80% train / 10% val / 10% test  —  user-level)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    best_ep = val_history.index(min(val_history)) + 1
    ax.axvline(best_ep, color='red', linestyle='--', alpha=0.5,
               label=f'Best val epoch {best_ep}')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Best val loss : {min(val_history):.4f}  at epoch {best_ep}")
    print(f"Final train loss : {train_history[-1]:.4f}")
    print(f"Final val loss   : {val_history[-1]:.4f}")
else:
    print("No history yet — run the training cell first.")

# ── 2. Test-set embedding sanity check ────────────────────────────────────────
# Load the best model weights, then check intra-user < inter-user distance
# on the held-out TEST users (10% never seen during training or val selection).
print("\n── Test-set sanity check (10% held-out users) ──")

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

n_check = min(20, len(test_idx))
intra_dists, inter_dists = [], []

with h5py.File(H5_PATH, 'r') as hf:
    full_idx_map = hf['index_map'][:]
    seqs_ds      = hf['sequences']

    with torch.no_grad():
        for i in range(n_check):
            u     = int(test_idx[i])
            start, end = full_idx_map[u]
            if end - start < 2:
                continue

            s1 = torch.from_numpy(seqs_ds[int(start)    ][None].astype(np.float32)).to(device)
            s2 = torch.from_numpy(seqs_ds[int(start) + 1][None].astype(np.float32)).to(device)
            e1 = model.forward_one(s1)
            e2 = model.forward_one(s2)
            intra_dists.append((e1 - e2).pow(2).sum().item())

            other   = int(test_idx[(i + 1) % n_check])
            o_start = int(full_idx_map[other, 0])
            eo      = model.forward_one(
                torch.from_numpy(seqs_ds[o_start][None].astype(np.float32)).to(device)
            )
            inter_dists.append((e1 - eo).pow(2).sum().item())

avg_intra = sum(intra_dists) / len(intra_dists)
avg_inter = sum(inter_dists) / len(inter_dists)
print(f"  Avg intra-user dist (same person)      : {avg_intra:.4f}")
print(f"  Avg inter-user dist (different person) : {avg_inter:.4f}")
sep = avg_inter / avg_intra if avg_intra > 0 else float('inf')
print(f"  Separation ratio (inter/intra)         : {sep:.2f}x")
print("  (higher ratio = better discrimination)")


## 9. Save and Export the Model  
The best model is already auto-saved to Drive during training. This cell verifies it, prints the final Drive layout, and downloads `typenet_pretrained.pth` directly to your browser.

In [ ]:

# ── Verify model file ──────────────────────────────────────────────────────────
assert os.path.exists(MODEL_PATH), f"Model not found at {MODEL_PATH}"
print(f"Model file : {MODEL_PATH}")
print(f"Size       : {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")

# Quick load test
state = torch.load(MODEL_PATH, map_location='cpu')
test_model = TypeNet()
test_model.load_state_dict(state)
test_model.eval()
dummy = torch.zeros(1, SEQ_LEN, INPUT_SIZE)
emb   = test_model.forward_one(dummy)
print(f"Embedding shape : {tuple(emb.shape)}  (expected [1, 128])")
print(f"Embedding norm  : {emb.norm().item():.4f}  (expected ≈ 1.0 — L2 normalised)")

# ── Show Drive layout ──────────────────────────────────────────────────────────
print(f"\nFiles in {DRIVE_BASE}:")
for f_name in sorted(os.listdir(DRIVE_BASE)):
    fpath = os.path.join(DRIVE_BASE, f_name)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {f_name:<40}  {size:9.1f} MB")

# ── Download model to local machine ───────────────────────────────────────────
# Uncomment if you want to download directly to your local browser:
# files.download(MODEL_PATH)

print("\n✅ Model is saved to Google Drive and ready to copy into the keystroke-service.")
print("   Place it at:  apps/services/keystroke-service/models/typenet_pretrained.pth")
